## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import CLIPForImageClassification
from datasets import load_from_disk, load_dataset
import numpy as np
import copy

## Configuration

In [ ]:
# Custom Config
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]

# Standard
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "CLIP_ViT_Vision"

## Class Prep

In [ ]:
def collate_fn(batch):
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/clip/modeling_clip.py
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(self.model.config.vision_config.hidden_size, num_classes)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage
    
    def forward(self, images):
        hidden_states = self.model.vision_model.embeddings(images)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        batch_size, seq_len, _ = hidden_states.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=hidden_states.device)
        attention_mask = attention_mask[:, None, None, :]
        casual_attention_mask = None

        for i, encoder_layer in enumerate(self.model.vision_model.encoder.layers):
            layer_outputs = encoder_layer(hidden_states, attention_mask, casual_attention_mask, output_attentions=False)
            hidden_states = layer_outputs[0]
            if i == self.transform_stage:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                cls = hidden_states[:, 0, :]
                cls = cls @ self.W
                hidden_states[:, 0, :] = cls
                break
            cls = hidden_states[:, 0, :]
        
        hidden_states = self.model.vision_model.post_layernorm(hidden_states[:, 0, :])
        logits = self.classifier(hidden_states)

        return logits, cls

In [ ]:
# https://huggingface.co/openai/clip-vit-base-patch32
refer = CLIPForImageClassification.from_pretrained("openai/clip-vit-base-patch32")

## Fine-Tune Prep

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
EPOCHS = 100

In [ ]:
for k in range(len(dataset_name)):
    num_class = [47, 10, 43, 10, 45, 196, 397, 10]
    dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
    num_classes = num_class[k]
    folder_path = f"./Reverse_Probe/{dataset_name[k]}"
    os.makedirs(folder_path, exist_ok=True)
    train = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/train_processed')
    val = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/val_processed')
    test = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/test_processed')

    train.set_format(type='torch', columns=["image", "label", "pixel_values"])
    val.set_format(type='torch', columns=["image", "label", "pixel_values"])
    test.set_format(type='torch', columns=["image", "label", "pixel_values"])

    train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    val_loader = DataLoader(val, batch_size=64, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

    for i in range(1,6):
        base = Augmented(copy.deepcopy(refer)).to(device)
        optimizer = torch.optim.AdamW([
            {'params': base.model.parameters(), 'lr': 1e-5}], weight_decay=0.01)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * EPOCHS)

        for param in base.classifier.parameters():
            param.requires_grad = False

        best_val_loss = float('inf')
        best_epoch = -1

        for epoch in range(EPOCHS):
            print(f"Epoch {epoch}/{EPOCHS} - Best Val Loss: {best_val_loss:.4f}, (Epoch {best_epoch})")

            base.train()
            total_train_loss = 0
            train_steps = 0
    
            for batch in tqdm(train_loader, desc="Training"):
                optimizer.zero_grad()
    
                images = batch["pixel_values"].to(device, non_blocking=True)
                labels = batch["labels"].to(device, non_blocking=True)
    
                logits, _ = base(images)
                loss = criterion(logits, labels)
    
                loss.backward()
                optimizer.step()
    
                total_train_loss += loss.item()
                train_steps += 1
    
            avg_train_loss = total_train_loss / train_steps
    
            # Validation
            base.eval()
            correct = 0
            total = 0
            total_val_loss = 0
            val_steps = 0
    
            with torch.no_grad():
                for batch in tqdm(val_loader, desc="Validation"):
                    images = batch["pixel_values"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)
    
                    logits, _ = base(images)
                    loss = criterion(logits, labels)
    
                    preds = torch.argmax(logits, dim=1)
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)
    
                    total_val_loss += loss.item()
                    val_steps += 1
    
            avg_val_loss = total_val_loss / val_steps
            val_acc = correct / total
    
            print(f"[Epoch {epoch}] Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")
    
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_epoch = epoch
                full_path = os.path.join(folder_path, f"best_{model_name}_{dataset_name[k]}_Reverse_Probe_{i}.pt")
                torch.save(base.state_dict(), full_path)

            scheduler.step()

In [ ]:
for k in range(len(dataset_name)):
    num_classes = num_class[k]
    folder_path = f"./Models/Reverse_Probe/Fine_Tuned/{dataset_name[k]}"

    base = Augmented(copy.deepcopy(refer)).to(device).eval()

    fine_tuned = {}
    for i in range(1,6):
        full_path = os.path.join(folder_path, f"best_{model_name}_{dataset_name[k]}_Reverse_Probe_{i}.pt")
        f_t = Augmented(copy.deepcopy(refer))
        f_t.load_state_dict(torch.load(full_path, map_location=device))
        model = Augmented(f_t.model, f_t.classifier).to(device).eval()
        model = model.eval().to(device)
        fine_tuned[i] = model
    
    correct_base = 0
    correct_fine_tuned = {i: 0 for i in range(1,6)}
    total = 0

    base_loss = 0
    fine_tuned_loss = {i: 0 for i in range(1,6)}
    total_loss = 0

    test = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/test_processed')
    test.set_format(type='torch', columns=["image", "label", "pixel_values"])
    test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        total += labels.size(0)
        total_loss += 1

        logits_base, _ = base(images)
        base_loss += criterion(logits_base, labels).item()
        pred = logits_base.argmax(dim=1)
        correct_base += (pred == labels).sum().item()
        
        for i in range(1,6):
            logits_fine_tuned, _ = fine_tuned[i](images)
            fine_tuned_loss[i] += criterion(logits_fine_tuned, labels).item()
            pred = logits_fine_tuned.argmax(dim=1)
            correct_fine_tuned[i] += (pred == labels).sum().item()
    
    arr = []
    avg_base_loss = base_loss / total_loss
    accuracy_base = correct_base / total
    print(f"\nAverage base loss: {avg_base_loss}, Base Accuracy: {accuracy_base}")
    for i in range(1,6):
        avg_best_loss = fine_tuned_loss[i] / total_loss
        accuracy_best = correct_fine_tuned[i] / total
        arr.append(accuracy_best)
        print(f"Average best loss: {avg_best_loss}, Best Accuracy: {accuracy_best}")
    print(arr)

In [ ]:
#  find /workspace -mindepth 1 -exec rm -rf {} +